In [3]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl (8.2 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)
Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl (36.6 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ -

In [4]:
# Cell 1 — Independent setup, dataset checks, and split configuration.
#
# Read the saved manifest rather than recreating feature settings.
# Keep model inputs separate from targets and metadata.
#
# The dates below are explicit research settings.
# They are not embedded inside model or data-processing functions.
#
# This cell reads metadata only. It does not load the full dataset,
# train models, or evaluate holdout performance.

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

try:
    import sklearn
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "scikit-learn is required for this notebook. "
        "Run %pip install scikit-learn in a separate cell, "
        "restart this notebook's kernel, then rerun Cell 1."
    ) from error


def find_project_root(start_path):
    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / ".git").exists()
            and (candidate / "raw_data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project. "
        "Open this notebook from the project's notebooks folder."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
REPORT_TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
MODEL_DIR = PROJECT_ROOT / "models"

manifest_candidates = sorted(
    REPORT_TABLES_DIR.glob("*_features_targets_*_manifest.json")
)

# Do not silently select between multiple datasets or instruments.
if len(manifest_candidates) != 1:
    raise ValueError(
        "Expected exactly one features/targets manifest. Found: "
        f"{[path.name for path in manifest_candidates]}"
    )

FEATURE_TARGET_MANIFEST_FILE = manifest_candidates[0]

with FEATURE_TARGET_MANIFEST_FILE.open(
    "r", encoding="utf-8"
) as handle:
    dataset_manifest = json.load(handle)


assert dataset_manifest["status"] in {
    "PASS",
    "PASS_WITH_COVERAGE_WARNINGS",
}

assert all(
    value is True
    for value in dataset_manifest["validation"].values()
), "The saved build contains an unsuccessful validation check."


instrument_config = dataset_manifest["instrument_config"]
feature_settings = dataset_manifest["feature_settings"]
target_settings = dataset_manifest["target_settings"]

MODEL_FEATURE_COLUMNS = list(
    dataset_manifest["column_roles"]["model_inputs"]
)
TARGET_COLUMNS = list(
    dataset_manifest["column_roles"]["targets_and_label_audit"]
)
METADATA_COLUMNS = list(
    dataset_manifest["column_roles"]["metadata"]
)

assert MODEL_FEATURE_COLUMNS
assert len(MODEL_FEATURE_COLUMNS) == len(set(MODEL_FEATURE_COLUMNS))

assert not set(MODEL_FEATURE_COLUMNS).intersection(TARGET_COLUMNS)
assert not set(MODEL_FEATURE_COLUMNS).intersection(METADATA_COLUMNS)
assert not set(TARGET_COLUMNS).intersection(METADATA_COLUMNS)


PRIMARY_HORIZON_MINUTES = int(
    target_settings["primary_horizon_minutes"]
)
AUXILIARY_HORIZON_MINUTES = int(
    target_settings["auxiliary_horizon_minutes"]
)

assert PRIMARY_HORIZON_MINUTES == int(
    instrument_config["forecast_horizon_minutes"]
)

DIRECTION_COLUMN = (
    f"target_direction_{PRIMARY_HORIZON_MINUTES}m"
)
AUXILIARY_RETURN_COLUMN = (
    f"target_return_bps_{AUXILIARY_HORIZON_MINUTES}m"
)

assert DIRECTION_COLUMN in TARGET_COLUMNS
assert AUXILIARY_RETURN_COLUMN in TARGET_COLUMNS


FEATURE_TARGET_INPUT_FILE = (
    PROJECT_ROOT / dataset_manifest["output"]["relative_path"]
)

assert FEATURE_TARGET_INPUT_FILE.is_file()
assert (
    FEATURE_TARGET_INPUT_FILE.stat().st_size
    == dataset_manifest["output"]["size_bytes"]
), "Dataset size differs from the saved manifest."


# Verify the footer and exact saved column order.
# Full row-value verification was performed in Notebook 03.
expected_columns = [
    *METADATA_COLUMNS,
    *MODEL_FEATURE_COLUMNS,
    *TARGET_COLUMNS,
]

with pq.ParquetFile(FEATURE_TARGET_INPUT_FILE) as parquet_file:
    dataset_row_count = parquet_file.metadata.num_rows
    saved_columns = parquet_file.schema_arrow.names

assert dataset_row_count == dataset_manifest["output"]["rows"]
assert saved_columns == expected_columns


# Explicit chronological boundaries.
# A decision exactly on a boundary belongs to the later period.
SPLIT_SETTINGS = {
    "model_selection_start_utc": "2023-01-01T00:00:00+00:00",
    "calibration_start_utc": "2024-01-01T00:00:00+00:00",
    "holdout_start_utc": "2025-01-01T00:00:00+00:00",
}

MODEL_SELECTION_START = pd.Timestamp(
    SPLIT_SETTINGS["model_selection_start_utc"]
)
CALIBRATION_START = pd.Timestamp(
    SPLIT_SETTINGS["calibration_start_utc"]
)
HOLDOUT_START = pd.Timestamp(
    SPLIT_SETTINGS["holdout_start_utc"]
)

first_decision = pd.Timestamp(
    dataset_manifest["output"]["first_decision_time_utc"]
)
last_decision = pd.Timestamp(
    dataset_manifest["output"]["last_decision_time_utc"]
)

assert (
    first_decision
    < MODEL_SELECTION_START
    < CALIBRATION_START
    < HOLDOUT_START
    < last_decision
), "Split boundaries are incompatible with dataset coverage."


# In the next cell, purge earlier-period rows whose target_time
# reaches or crosses the next period's starting boundary.
#
# Past-only feature history may extend into an earlier period.
# That mirrors live operation and is not future-data leakage.
#
# No scalers, imputers, or models have been fitted here.

print("Notebook setup: PASS")
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("scikit-learn:", sklearn.__version__)
print()
print("Instrument:", instrument_config["symbol"])
print(f"Dataset rows: {dataset_row_count:,}")
print("Model-input features:", len(MODEL_FEATURE_COLUMNS))
print("Primary horizon:", PRIMARY_HORIZON_MINUTES, "minutes")
print("Auxiliary horizon:", AUXILIARY_HORIZON_MINUTES, "minutes")
print()
print("Training:        before", MODEL_SELECTION_START)
print("Model selection: from", MODEL_SELECTION_START)
print("Calibration:     from", CALIBRATION_START)
print("Holdout:         from", HOLDOUT_START)
print()
print("Input:", FEATURE_TARGET_INPUT_FILE)
print("No models fitted; holdout performance remains unevaluated.")

Notebook setup: PASS
Python: 3.13.16
Pandas: 3.0.6
PyArrow: 25.0.1
scikit-learn: 1.9.1

Instrument: EURUSD
Dataset rows: 5,672,911
Model-input features: 15
Primary horizon: 10 minutes
Auxiliary horizon: 5 minutes

Training:        before 2023-01-01 00:00:00+00:00
Model selection: from 2023-01-01 00:00:00+00:00
Calibration:     from 2024-01-01 00:00:00+00:00
Holdout:         from 2025-01-01 00:00:00+00:00

Input: C:\Users\Ijzd4\trend models\EURUSD 15\data\processed\eurusd_features_targets_10m_2010-01-01_2026-10-06.parquet
No models fitted; holdout performance remains unevaluated.


In [5]:
# Cell 2 — Create chronological splits and purge crossing targets.
#
# Each observation belongs to a period based on decision_time.
# Remove earlier-period rows whose target ends at or after
# the next period's starting boundary.
#
# Check BOTH the primary and auxiliary target endpoints.
# Past-only feature history may cross a period boundary:
# it was already observable at decision time.
#
# No random splitting, model fitting, or performance evaluation.

SPLIT_DEFINITIONS = {
    "training": (None, MODEL_SELECTION_START),
    "model_selection": (MODEL_SELECTION_START, CALIBRATION_START),
    "calibration": (CALIBRATION_START, HOLDOUT_START),
    "holdout": (HOLDOUT_START, None),
}

# Code 0 identifies purged observations.
SPLIT_CODES = {
    "training": 1,
    "model_selection": 2,
    "calibration": 3,
    "holdout": 4,
}


split_timeline = pd.read_parquet(
    FEATURE_TARGET_INPUT_FILE,
    columns=[
        "decision_time",
        "target_time",
        "auxiliary_target_time",
    ],
)

assert len(split_timeline) == dataset_row_count
assert split_timeline.index.equals(pd.RangeIndex(dataset_row_count))
assert not split_timeline.isna().any().any()

decision_times = split_timeline["decision_time"]

assert decision_times.is_monotonic_increasing
assert decision_times.is_unique
assert decision_times.iloc[0] == first_decision
assert decision_times.iloc[-1] == last_decision

assert (
    split_timeline["target_time"]
    == decision_times + pd.Timedelta(minutes=PRIMARY_HORIZON_MINUTES)
).all()

assert (
    split_timeline["auxiliary_target_time"]
    == decision_times + pd.Timedelta(minutes=AUXILIARY_HORIZON_MINUTES)
).all()


split_id = pd.Series(
    np.zeros(dataset_row_count, dtype="int8"),
    index=split_timeline.index,
    name="split_id",
)

split_masks = {}
split_summary_rows = []


for role, (period_start, period_end) in SPLIT_DEFINITIONS.items():
    period_mask = pd.Series(True, index=split_timeline.index)

    if period_start is not None:
        period_mask &= decision_times.ge(period_start)

    if period_end is not None:
        period_mask &= decision_times.lt(period_end)

    retained_mask = period_mask.copy()

    if period_end is not None:
        retained_mask &= (
            split_timeline["target_time"].lt(period_end)
            & split_timeline["auxiliary_target_time"].lt(period_end)
        )

    before_purge = int(period_mask.sum())
    retained_count = int(retained_mask.sum())

    assert retained_count > 0, (
        f"No eligible observations in {role}. "
        "Review the configured dates."
    )

    # Check that no observation is assigned to two periods.
    assert split_id.loc[retained_mask].eq(0).all()

    split_id.loc[retained_mask] = SPLIT_CODES[role]
    split_masks[role] = retained_mask

    retained_times = split_timeline.loc[retained_mask]

    last_label_end = max(
        retained_times["target_time"].iloc[-1],
        retained_times["auxiliary_target_time"].iloc[-1],
    )

    if period_end is not None:
        assert last_label_end < period_end

    split_summary_rows.append({
        "period": role,
        "rows_before_purge": before_purge,
        "purged_rows": before_purge - retained_count,
        "retained_rows": retained_count,
        "first_decision_utc": retained_times["decision_time"].iloc[0],
        "last_decision_utc": retained_times["decision_time"].iloc[-1],
        "last_label_end_utc": last_label_end,
    })


split_summary = pd.DataFrame(split_summary_rows)

assert int(
    split_summary["rows_before_purge"].sum()
) == dataset_row_count

purged_row_count = int(split_id.eq(0).sum())

assert purged_row_count == int(
    split_summary["purged_rows"].sum()
)

assert (
    int(split_summary["retained_rows"].sum()) + purged_row_count
    == dataset_row_count
)


# Independently check separation between adjacent retained periods.
summary_by_period = split_summary.set_index("period")
period_names = list(SPLIT_DEFINITIONS)

for earlier, later in zip(period_names[:-1], period_names[1:]):
    assert (
        summary_by_period.loc[earlier, "last_label_end_utc"]
        < summary_by_period.loc[later, "first_decision_utc"]
    ), f"Target overlap between {earlier} and {later}."


for role, code in SPLIT_CODES.items():
    assert split_id.eq(code).equals(split_masks[role])


print("Chronological split assignment: PASS")
print("Target-boundary purging: PASS")
print("Period separation: PASS")
print("Row reconciliation: PASS")
print(f"Total purged rows: {purged_row_count:,}")
print()
display(split_summary)

print(
    "Zero purged rows can be valid when market closures already "
    "separate the periods."
)
print("Calibration and holdout performance remain unevaluated.")


# Retain split_timeline, split_id, split_masks, and split_summary.
# They will control which observations each later stage may use.
del period_mask, retained_mask, retained_times

Chronological split assignment: PASS
Target-boundary purging: PASS
Period separation: PASS
Row reconciliation: PASS
Total purged rows: 0



,period,rows_before_purge,purged_rows,retained_rows,first_decision_utc,last_decision_utc,last_label_end_utc
0,training,4440160,0,4440160,2010-01-01 01:01:00+00:00,2022-12-30 21:50:00+00:00,2022-12-30 22:00:00+00:00
1,model_selection,329184,0,329184,2023-01-02 00:19:00+00:00,2023-12-29 21:50:00+00:00,2023-12-29 22:00:00+00:00
2,calibration,316661,0,316661,2024-01-01 23:38:00+00:00,2024-12-31 21:50:00+00:00,2024-12-31 22:00:00+00:00
3,holdout,586906,0,586906,2025-01-01 23:05:00+00:00,2026-10-06 20:29:00+00:00,2026-10-06 20:39:00+00:00


Zero purged rows can be valid when market closures already separate the periods.
Calibration and holdout performance remain unevaluated.


In [6]:
# Cell 3 — Load development data, fit baselines, and evaluate on 2023.
#
# A. Verify dataset identity.
# B. Load ONLY training and model-selection observations.
# C. Fit training-derived benchmarks and two initial models.
# D. Evaluate ONLY on the model-selection period.
#
# Calibration and holdout data are not supplied to these models.
# Initial model parameters are explicit and are not optimized.
#
# The return estimator predicts signed, gross midpoint movement
# in basis points. It does NOT yet predict executable net profit.

import hashlib
import time

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
)
from threadpoolctl import threadpool_limits


# A — Verify that we are using the exact previously validated file.
def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(8 * 1024 * 1024)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


input_stat = FEATURE_TARGET_INPUT_FILE.stat()
MODEL_INPUT_SNAPSHOT = (
    input_stat.st_size,
    input_stat.st_mtime_ns,
)

print("Verifying dataset checksum...", flush=True)

MODEL_INPUT_SHA256 = file_sha256(FEATURE_TARGET_INPUT_FILE)

assert MODEL_INPUT_SHA256 == dataset_manifest["output"]["sha256"], (
    "Dataset bytes differ from the validated manifest."
)

print("Dataset identity: PASS", flush=True)


# B — Load only the two development periods.
# Explicit date filters also enforce the target-boundary purge.
# PyArrow applies the filters to individual rows.
def load_development_period(role):
    if role not in {"training", "model_selection"}:
        raise ValueError(
            "This loader intentionally refuses calibration "
            "and holdout periods."
        )

    period_start, period_end = SPLIT_DEFINITIONS[role]
    filters = []

    if period_start is not None:
        filters.append(
            ("decision_time", ">=", period_start.to_pydatetime())
        )

    if period_end is not None:
        boundary = period_end.to_pydatetime()
        filters.extend([
            ("decision_time", "<", boundary),
            ("target_time", "<", boundary),
            ("auxiliary_target_time", "<", boundary),
        ])

    frame = pd.read_parquet(
        FEATURE_TARGET_INPUT_FILE,
        engine="pyarrow",
        columns=[
            "decision_time",
            *MODEL_FEATURE_COLUMNS,
            DIRECTION_COLUMN,
            AUXILIARY_RETURN_COLUMN,
        ],
        filters=filters,
    )

    expected_times = split_timeline.loc[
        split_masks[role], "decision_time"
    ].reset_index(drop=True)

    assert frame["decision_time"].reset_index(drop=True).equals(
        expected_times
    ), f"Loaded rows do not match the validated {role} split."

    assert not frame.isna().any().any()

    # Float32 reduces input-matrix memory.
    # Record this dtype so later inference uses the same convention.
    X = frame[MODEL_FEATURE_COLUMNS].to_numpy(
        dtype="float32", copy=True
    )
    direction_y = frame[DIRECTION_COLUMN].to_numpy(
        dtype="int8", copy=True
    )
    return_y = frame[AUXILIARY_RETURN_COLUMN].to_numpy(
        dtype="float64", copy=True
    )

    assert X.shape == (len(expected_times), len(MODEL_FEATURE_COLUMNS))
    assert np.isfinite(X).all()
    assert np.isfinite(return_y).all()

    return X, direction_y, return_y


print("Loading training data...", flush=True)

X_training, y_training_direction, y_training_return = (
    load_development_period("training")
)

print("Loading model-selection data...", flush=True)

X_selection, y_selection_direction, y_selection_return = (
    load_development_period("model_selection")
)


CLASS_LABELS = np.array(
    sorted(target_settings["direction_encoding"].values()),
    dtype="int8",
)

CLASS_NAMES = {
    int(code): name
    for name, code in target_settings["direction_encoding"].items()
}

assert len(CLASS_LABELS) == 3
assert len(set(CLASS_LABELS.tolist())) == 3

np.testing.assert_array_equal(
    np.unique(y_training_direction), CLASS_LABELS
)

assert np.isin(y_selection_direction, CLASS_LABELS).all()
assert np.var(y_training_return) > 0
assert np.var(y_selection_return) > 0

print("Development-data alignment: PASS")
print(f"Training rows: {len(X_training):,}")
print(f"Model-selection rows: {len(X_selection):,}")
print(
    "Combined feature-matrix memory:",
    f"{(X_training.nbytes + X_selection.nbytes) / 1024**2:.2f} MiB",
)
print("Model internals will require additional memory.")
print()


# C — Explicit initial settings.
# Small trees and large minimum leaves provide a conservative
# starting model capacity, not a guarantee against overfitting.
#
# No class balancing: retain the training class frequencies.
# No feature scaler is fitted for these tree models.
# No target clipping or removal of extreme market events.

TRAINING_SETTINGS = {
    "input_dtype": "float32",
    "thread_limit": 4,
    "training_sample": "all_retained_training_rows",
}

TREE_SETTINGS = {
    "learning_rate": 0.05,
    "max_iter": 100,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 1000,
    "l2_regularization": 1.0,
    "categorical_features": None,
    "early_stopping": False,
    "validation_fraction": None,
    "random_state": 42,
    "verbose": 2,
}

prior_classifier = DummyClassifier(strategy="prior")

direction_model = HistGradientBoostingClassifier(
    loss="log_loss",
    class_weight=None,
    **TREE_SETTINGS,
)

return_model = HistGradientBoostingRegressor(
    loss="squared_error",
    **TREE_SETTINGS,
)

# Constant-return benchmarks use no model-selection information.
TRAINING_MEAN_RETURN_BPS = float(y_training_return.mean())

fit_timings = {}

with threadpool_limits(limits=TRAINING_SETTINGS["thread_limit"]):
    prior_classifier.fit(X_training, y_training_direction)

    print(
        "\nFitting the primary direction model...",
        flush=True,
    )
    started = time.perf_counter()

    direction_model.fit(X_training, y_training_direction)

    fit_timings["direction_model_minutes"] = (
        time.perf_counter() - started
    ) / 60

    print(
        "\nFitting the auxiliary gross-return model...",
        flush=True,
    )
    started = time.perf_counter()

    return_model.fit(X_training, y_training_return)

    fit_timings["return_model_minutes"] = (
        time.perf_counter() - started
    ) / 60

    # Predict only on the 2023 model-selection period.
    prior_probabilities = prior_classifier.predict_proba(X_selection)
    selection_probabilities = direction_model.predict_proba(X_selection)
    selection_return_predictions = return_model.predict(X_selection)


# Confirm there was no automatic internal early-stopping split.
assert not direction_model.do_early_stopping_
assert not return_model.do_early_stopping_

assert direction_model.n_iter_ == TREE_SETTINGS["max_iter"]
assert return_model.n_iter_ == TREE_SETTINGS["max_iter"]

# Probability columns must follow the documented class ordering.
np.testing.assert_array_equal(
    prior_classifier.classes_, CLASS_LABELS
)
np.testing.assert_array_equal(
    direction_model.classes_, CLASS_LABELS
)

assert np.isfinite(selection_return_predictions).all()


# D — Evaluate probabilistic classification against the prior baseline.
def classification_metrics_row(name, probabilities):
    assert probabilities.shape == (
        len(y_selection_direction), len(CLASS_LABELS)
    )
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

    np.testing.assert_allclose(
        probabilities.sum(axis=1),
        1.0,
        rtol=1e-7,
        atol=1e-7,
    )

    predictions = CLASS_LABELS[np.argmax(probabilities, axis=1)]
    one_hot_truth = (
        y_selection_direction[:, None] == CLASS_LABELS[None, :]
    )

    # Multiclass Brier here means the mean SUM of squared
    # probability errors across classes, with range [0, 2].
    return {
        "model": name,
        "accuracy": float(
            accuracy_score(y_selection_direction, predictions)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_selection_direction, predictions)
        ),
        "macro_f1": float(
            f1_score(
                y_selection_direction,
                predictions,
                labels=CLASS_LABELS,
                average="macro",
                zero_division=0,
            )
        ),
        "log_loss": float(
            log_loss(
                y_selection_direction,
                probabilities,
                labels=CLASS_LABELS,
            )
        ),
        "multiclass_brier_sum": float(
            np.mean(
                np.sum(
                    (probabilities - one_hot_truth) ** 2,
                    axis=1,
                )
            )
        ),
    }


classification_metrics = pd.DataFrame([
    classification_metrics_row(
        "training_class_prior", prior_probabilities
    ),
    classification_metrics_row(
        "hist_gradient_boosting", selection_probabilities
    ),
])


# Evaluate gross-return estimates against zero and training mean.
def regression_metrics_row(name, predictions):
    assert predictions.shape == y_selection_return.shape
    assert np.isfinite(predictions).all()

    return {
        "model": name,
        "mae_bps": float(
            mean_absolute_error(y_selection_return, predictions)
        ),
        "rmse_bps": float(
            np.sqrt(
                mean_squared_error(y_selection_return, predictions)
            )
        ),
        "r2": float(
            r2_score(y_selection_return, predictions)
        ),
        "mean_prediction_bps": float(predictions.mean()),
        "mean_error_bps": float(
            (predictions - y_selection_return).mean()
        ),
    }


regression_metrics = pd.DataFrame([
    regression_metrics_row(
        "always_zero",
        np.zeros_like(y_selection_return),
    ),
    regression_metrics_row(
        "training_mean",
        np.full_like(
            y_selection_return, TRAINING_MEAN_RETURN_BPS
        ),
    ),
    regression_metrics_row(
        "hist_gradient_boosting",
        selection_return_predictions,
    ),
])


# Show performance for each direction rather than relying on accuracy.
selection_direction_predictions = CLASS_LABELS[
    np.argmax(selection_probabilities, axis=1)
]

precision, recall, class_f1, support = (
    precision_recall_fscore_support(
        y_selection_direction,
        selection_direction_predictions,
        labels=CLASS_LABELS,
        zero_division=0,
    )
)

direction_class_report = pd.DataFrame({
    "direction": [CLASS_NAMES[int(code)] for code in CLASS_LABELS],
    "precision": precision,
    "recall": recall,
    "f1": class_f1,
    "observations": support,
})

selection_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        y_selection_direction,
        selection_direction_predictions,
        labels=CLASS_LABELS,
    ),
    index=[
        f"actual_{CLASS_NAMES[int(code)]}" for code in CLASS_LABELS
    ],
    columns=[
        f"predicted_{CLASS_NAMES[int(code)]}" for code in CLASS_LABELS
    ],
)


# Confirm the input file did not change during fitting.
input_stat_after = FEATURE_TARGET_INPUT_FILE.stat()

assert (
    input_stat_after.st_size,
    input_stat_after.st_mtime_ns,
) == MODEL_INPUT_SNAPSHOT


print("\nFit and evaluation pipeline checks: PASS")
print("Evaluation period: model selection only — 2023")
print()

print("Classification — higher accuracy/F1, lower loss/Brier is better:")
display(classification_metrics.round(6))

print("Gross-return regression — lower MAE/RMSE is better:")
display(regression_metrics.round(6))

print("Direction-specific classification results:")
display(direction_class_report.round(6))

print("Confusion matrix:")
display(selection_confusion_matrix)

print("Fit timings:")
display(pd.DataFrame([fit_timings]).round(2))

print("Probabilities are NOT calibrated yet.")
print("No trading gates or executable P&L have been evaluated.")
print("Calibration and holdout performance remain untouched.")
print(
    "These are overlapping forecast observations, "
    "not independent trades."
)


# Keep fitted models, predictions, settings, and metrics.
# A later cell will save a verified experiment checkpoint.

Verifying dataset checksum...
Dataset identity: PASS
Loading training data...
Loading model-selection data...
Development-data alignment: PASS
Training rows: 4,440,160
Model-selection rows: 329,184
Combined feature-matrix memory: 272.90 MiB
Model internals will require additional memory.


Fitting the primary direction model...
Binning 0.533 GB of training data: 1.282 s
Fitting gradient boosted rounds:
[1/100] 3 trees, 45 leaves (15 on avg), max depth = 7, in 0.624s
[2/100] 3 trees, 45 leaves (15 on avg), max depth = 5, in 0.571s
[3/100] 3 trees, 45 leaves (15 on avg), max depth = 7, in 0.552s
[4/100] 3 trees, 45 leaves (15 on avg), max depth = 6, in 0.563s
[5/100] 3 trees, 45 leaves (15 on avg), max depth = 7, in 0.537s
[6/100] 3 trees, 45 leaves (15 on avg), max depth = 6, in 0.551s
[7/100] 3 trees, 45 leaves (15 on avg), max depth = 7, in 0.510s
[8/100] 3 trees, 45 leaves (15 on avg), max depth = 7, in 0.553s
[9/100] 3 trees, 45 leaves (15 on avg), max depth = 8, in 0.470s
[10/100] 

,model,accuracy,balanced_accuracy,macro_f1,log_loss,multiclass_brier_sum
0,training_class_prior,0.712018,0.333333,0.277263,0.800618,0.451964
1,hist_gradient_boosting,0.712018,0.333333,0.277263,0.790625,0.446158


Gross-return regression — lower MAE/RMSE is better:


,model,mae_bps,rmse_bps,r2,mean_prediction_bps,mean_error_bps
0,always_zero,1.915085,2.908988,-0.000000,0.000000,-0.000888
1,training_mean,1.915122,2.908990,-0.000001,-0.002037,-0.002925
2,hist_gradient_boosting,1.914532,2.908647,0.000234,-0.001442,-0.002331


Direction-specific classification results:


...

Confusion matrix:


,predicted_DOWN,predicted_FLAT,predicted_UP
actual_DOWN,0,47010,0
actual_FLAT,0,234385,0
actual_UP,0,47789,0


Fit timings:


,direction_model_minutes,return_model_minutes
0,0.72,0.16


Probabilities are NOT calibrated yet.
No trading gates or executable P&L have been evaluated.
Calibration and holdout performance remain untouched.
These are overlapping forecast observations, not independent trades.


In [7]:
# Cell 4 — Diagnose the first baseline without changing it.
#
# A. Compare training performance with model-selection performance.
# B. Inspect raw probability ranges.
# C. Examine actual returns across predicted-return bins.
#
# These are research diagnostics, not trades or executable P&L.
# Do not change label bands simply to make directional predictions appear.
# Keep calibration and holdout untouched.

AUDIT_BATCH_ROWS = 100_000

training_confusion = np.zeros(
    (len(CLASS_LABELS), len(CLASS_LABELS)),
    dtype="int64",
)

training_log_loss_sum = 0.0
training_brier_sum = 0.0
training_return_sse = 0.0
audited_training_rows = 0

audit_started = time.perf_counter()


# A — Calculate training scores in bounded-memory batches.
# This does not modify or refit either model.
with threadpool_limits(limits=TRAINING_SETTINGS["thread_limit"]):
    for start in range(0, len(X_training), AUDIT_BATCH_ROWS):
        stop = min(start + AUDIT_BATCH_ROWS, len(X_training))

        batch_probabilities = direction_model.predict_proba(
            X_training[start:stop]
        )
        batch_return_predictions = return_model.predict(
            X_training[start:stop]
        )

        batch_direction_truth = y_training_direction[start:stop]
        batch_return_truth = y_training_return[start:stop]

        assert np.isfinite(batch_probabilities).all()
        assert np.isfinite(batch_return_predictions).all()

        batch_direction_predictions = CLASS_LABELS[
            np.argmax(batch_probabilities, axis=1)
        ]

        training_confusion += confusion_matrix(
            batch_direction_truth,
            batch_direction_predictions,
            labels=CLASS_LABELS,
        )

        training_log_loss_sum += log_loss(
            batch_direction_truth,
            batch_probabilities,
            labels=CLASS_LABELS,
            normalize=False,
        )

        batch_one_hot = (
            batch_direction_truth[:, None]
            == CLASS_LABELS[None, :]
        )

        training_brier_sum += float(
            np.sum((batch_probabilities - batch_one_hot) ** 2)
        )

        training_return_sse += float(
            np.sum(
                (batch_return_predictions - batch_return_truth) ** 2
            )
        )

        audited_training_rows += stop - start

        if audited_training_rows % 1_000_000 == 0:
            print(
                f"Audited training rows: {audited_training_rows:,}",
                flush=True,
            )


assert audited_training_rows == len(X_training)
assert int(training_confusion.sum()) == len(X_training)

training_support = training_confusion.sum(axis=1)

training_recalls = np.divide(
    np.diag(training_confusion),
    training_support,
    out=np.zeros(len(CLASS_LABELS), dtype="float64"),
    where=training_support > 0,
)

training_return_mse = (
    training_return_sse / audited_training_rows
)

selection_class_scores = classification_metrics.set_index(
    "model"
).loc["hist_gradient_boosting"]

selection_return_scores = regression_metrics.set_index(
    "model"
).loc["hist_gradient_boosting"]


training_selection_comparison = pd.DataFrame([
    {
        "period": "training",
        "accuracy": float(
            np.trace(training_confusion) / audited_training_rows
        ),
        "balanced_accuracy": float(
            training_recalls[training_support > 0].mean()
        ),
        "log_loss": float(
            training_log_loss_sum / audited_training_rows
        ),
        "multiclass_brier_sum": float(
            training_brier_sum / audited_training_rows
        ),
        "return_rmse_bps": float(np.sqrt(training_return_mse)),
        "return_r2": float(
            1 - training_return_mse / np.var(y_training_return)
        ),
    },
    {
        "period": "model_selection_2023",
        "accuracy": float(selection_class_scores["accuracy"]),
        "balanced_accuracy": float(
            selection_class_scores["balanced_accuracy"]
        ),
        "log_loss": float(selection_class_scores["log_loss"]),
        "multiclass_brier_sum": float(
            selection_class_scores["multiclass_brier_sum"]
        ),
        "return_rmse_bps": float(
            selection_return_scores["rmse_bps"]
        ),
        "return_r2": float(selection_return_scores["r2"]),
    },
])


# B — Inspect probability variation on 2023 only.
# Better log loss does not necessarily mean directional trades
# are identifiable at the requested probability threshold.
probability_diagnostic_rows = []

for column_index, code in enumerate(CLASS_LABELS):
    probabilities = selection_probabilities[:, column_index]

    probability_diagnostic_rows.append({
        "direction": CLASS_NAMES[int(code)],
        "training_prior": float(
            prior_classifier.class_prior_[column_index]
        ),
        "mean_probability": float(probabilities.mean()),
        "observed_frequency": float(
            np.mean(y_selection_direction == code)
        ),
        "minimum": float(probabilities.min()),
        "median": float(np.median(probabilities)),
        "p95": float(np.quantile(probabilities, 0.95)),
        "maximum": float(probabilities.max()),
    })

probability_diagnostics = pd.DataFrame(
    probability_diagnostic_rows
)

directional_probability_counts = {}

for column_index, code in enumerate(CLASS_LABELS):
    name = CLASS_NAMES[int(code)]

    if name in {"UP", "DOWN"}:
        directional_probability_counts[name] = int(
            np.sum(
                selection_probabilities[:, column_index] > 0.70
            )
        )


# C — Group 2023 observations by their predicted gross return.
# Bins use predictions only; their actual outcomes are then summarized.
# This is a model-selection diagnostic, not a frozen trading policy.
#
# Duplicate boundaries may produce fewer than ten bins.
# A constant prediction produces one bin.
unique_prediction_count = np.unique(
    selection_return_predictions
).size

if unique_prediction_count > 1:
    prediction_bins = pd.qcut(
        pd.Series(selection_return_predictions),
        q=10,
        labels=False,
        duplicates="drop",
    )
else:
    prediction_bins = pd.Series(
        np.zeros(len(selection_return_predictions), dtype="int8")
    )

return_diagnostic_frame = pd.DataFrame({
    "prediction_bin": prediction_bins,
    "predicted_return_bps": selection_return_predictions,
    "actual_return_bps": y_selection_return,
})

assert not return_diagnostic_frame.isna().any().any()

return_ranking_diagnostics = (
    return_diagnostic_frame
    .groupby("prediction_bin", sort=True, observed=True)
    .agg(
        observations=("actual_return_bps", "size"),
        mean_prediction_bps=("predicted_return_bps", "mean"),
        mean_actual_bps=("actual_return_bps", "mean"),
        actual_std_bps=("actual_return_bps", "std"),
    )
    .reset_index()
)

assert int(
    return_ranking_diagnostics["observations"].sum()
) == len(X_selection)


# Ensure the underlying dataset remained unchanged.
current_input_stat = FEATURE_TARGET_INPUT_FILE.stat()

assert (
    current_input_stat.st_size,
    current_input_stat.st_mtime_ns,
) == MODEL_INPUT_SNAPSHOT


print("\nBaseline diagnostic calculations: PASS")

print("\nTraining versus model selection:")
display(training_selection_comparison.round(6))

print("\nRaw model-selection probabilities:")
display(probability_diagnostics.round(6))

print("\n2023 observations with raw directional probability >70%:")
print(directional_probability_counts)

print(
    "\nPredicted-return bins, lowest to highest "
    f"({AUXILIARY_HORIZON_MINUTES}-minute gross returns):"
)
display(return_ranking_diagnostics.round(6))

print(
    "\nDiagnostic elapsed time:",
    f"{(time.perf_counter() - audit_started) / 60:.2f} minutes",
)
print("No models refitted and no thresholds changed.")
print("Calibration and holdout performance remain untouched.")
print("Bin averages are not evidence of executable net profit.")


# Retain diagnostic tables for the experiment checkpoint.
del batch_probabilities, batch_return_predictions
del batch_direction_truth, batch_return_truth
del batch_direction_predictions, batch_one_hot
del return_diagnostic_frame, prediction_bins

Audited training rows: 1,000,000
Audited training rows: 2,000,000
Audited training rows: 3,000,000
Audited training rows: 4,000,000

Baseline diagnostic calculations: PASS

Training versus model selection:


,period,accuracy,balanced_accuracy,log_loss,multiclass_brier_sum,return_rmse_bps,return_r2
0,training,0.728327,0.333333,0.763359,0.427185,3.364360,0.002803
1,model_selection_2023,0.712018,0.333333,0.790625,0.446158,2.908647,0.000234



Raw model-selection probabilities:


,direction,training_prior,mean_probability,observed_frequency,minimum,median,p95,maximum
0,DOWN,0.136113,0.142744,0.142808,0.016511,0.142863,0.183148,0.281983
1,FLAT,0.728327,0.714518,0.712018,0.496690,0.713150,0.792823,0.968538
2,UP,0.135560,0.142738,0.145174,0.014951,0.142548,0.184623,0.319513



2023 observations with raw directional probability >70%:
{'DOWN': 0, 'UP': 0}

Predicted-return bins, lowest to highest (5-minute gross returns):


,prediction_bin,observations,mean_prediction_bps,mean_actual_bps,actual_std_bps
0,0,32921,-0.162955,-0.114802,3.954203
1,1,32916,-0.066041,-0.013517,2.979716
2,2,32929,-0.035388,-0.018957,2.621942
3,3,32908,-0.017710,-0.029975,2.444539
4,4,32920,-0.004290,0.035145,2.266944
5,5,32916,0.008410,-0.006068,2.331618
6,6,32919,0.021795,0.017063,2.345717
7,7,32918,0.038375,0.024986,2.495435
8,8,32921,0.062355,0.019340,2.807896
9,9,32916,0.141046,0.095671,4.128448



Diagnostic elapsed time: 0.42 minutes
No models refitted and no thresholds changed.
Calibration and holdout performance remain untouched.
Bin averages are not evidence of executable net profit.


In [8]:
# Cell 5 — Save and verify the first research baseline.
#
# Save two files:
# 1. A local model bundle containing fitted models and configuration.
# 2. A readable JSON report containing metrics and diagnostics.
#
# Do NOT save the large training arrays inside the model bundle.
# Do NOT mark these models as selected, calibrated, or trade-ready.
#
# Existing files are never overwritten automatically.
# Partial files remain available for inspection if a check fails.

import re
import joblib
from importlib.metadata import version


BASELINE_RESEARCH_STATUS = "BASELINE_ONLY_NOT_TRADE_READY"

symbol_slug = re.sub(
    r"[^a-z0-9_-]+",
    "_",
    str(instrument_config["symbol"]).lower(),
).strip("_")

BASELINE_EXPERIMENT_ID = (
    f"{symbol_slug}_baseline_h{PRIMARY_HORIZON_MINUTES}"
    f"_r{AUXILIARY_HORIZON_MINUTES}_v1"
)

BASELINE_MODEL_FILE = (
    MODEL_DIR / f"{BASELINE_EXPERIMENT_ID}.joblib"
)
TEMP_BASELINE_MODEL_FILE = (
    MODEL_DIR / f"{BASELINE_EXPERIMENT_ID}.partial.joblib"
)

BASELINE_REPORT_FILE = (
    REPORT_TABLES_DIR / f"{BASELINE_EXPERIMENT_ID}_report.json"
)
TEMP_BASELINE_REPORT_FILE = (
    REPORT_TABLES_DIR / f"{BASELINE_EXPERIMENT_ID}_report.partial.json"
)


for path in [
    BASELINE_MODEL_FILE,
    TEMP_BASELINE_MODEL_FILE,
    BASELINE_REPORT_FILE,
    TEMP_BASELINE_REPORT_FILE,
]:
    if path.exists():
        raise FileExistsError(
            f"Checkpoint already exists; inspect before rerunning: {path}"
        )


# Ensure the source specification has not changed since setup.
with FEATURE_TARGET_MANIFEST_FILE.open(
    "r", encoding="utf-8"
) as handle:
    assert json.load(handle) == dataset_manifest

current_stat = FEATURE_TARGET_INPUT_FILE.stat()

assert (
    current_stat.st_size,
    current_stat.st_mtime_ns,
) == MODEL_INPUT_SNAPSHOT


# Fingerprint parsed JSON, avoiding differences from line endings.
feature_manifest_fingerprint = hashlib.sha256(
    json.dumps(
        dataset_manifest,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
).hexdigest()


environment_versions = {
    "python": sys.version.split()[0],
    **{
        package: version(package)
        for package in [
            "numpy",
            "pandas",
            "pyarrow",
            "scipy",
            "scikit-learn",
            "joblib",
            "threadpoolctl",
        ]
    },
}


baseline_bundle = {
    "experiment_id": BASELINE_EXPERIMENT_ID,
    "research_status": BASELINE_RESEARCH_STATUS,
    "environment_versions": environment_versions,
    "dataset_sha256": MODEL_INPUT_SHA256,
    "feature_manifest_fingerprint": feature_manifest_fingerprint,
    "instrument_config": instrument_config,
    "feature_settings": feature_settings,
    "target_settings": target_settings,
    "split_settings": SPLIT_SETTINGS,
    "training_settings": TRAINING_SETTINGS.copy(),
    "feature_columns": MODEL_FEATURE_COLUMNS.copy(),
    "class_labels": [int(code) for code in CLASS_LABELS],
    "direction_target_column": DIRECTION_COLUMN,
    "return_target_column": AUXILIARY_RETURN_COLUMN,
    "return_prediction_units": "gross_midpoint_return_basis_points",
    "training_mean_return_bps": TRAINING_MEAN_RETURN_BPS,
    "probability_calibration_fitted": False,
    "holdout_performance_evaluated": False,
    "prior_classifier": prior_classifier,
    "direction_model": direction_model,
    "return_model": return_model,
}


MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLES_DIR.mkdir(parents=True, exist_ok=True)


# Write exclusively to a staging file.
# Load only this trusted file that we have just created ourselves.
with TEMP_BASELINE_MODEL_FILE.open("xb") as handle:
    joblib.dump(baseline_bundle, handle, compress=3)

restored_bundle = joblib.load(TEMP_BASELINE_MODEL_FILE)


assert restored_bundle["experiment_id"] == BASELINE_EXPERIMENT_ID
assert restored_bundle["research_status"] == BASELINE_RESEARCH_STATUS
assert restored_bundle["feature_columns"] == MODEL_FEATURE_COLUMNS
assert restored_bundle["dataset_sha256"] == MODEL_INPUT_SHA256
assert restored_bundle["environment_versions"] == environment_versions

assert not restored_bundle["probability_calibration_fitted"]
assert not restored_bundle["holdout_performance_evaluated"]

for key in ["direction_model", "return_model"]:
    assert restored_bundle[key].n_features_in_ == len(
        MODEL_FEATURE_COLUMNS
    )

np.testing.assert_array_equal(
    restored_bundle["direction_model"].classes_,
    CLASS_LABELS,
)

assert (
    restored_bundle["direction_model"].get_params()
    == direction_model.get_params()
)
assert (
    restored_bundle["return_model"].get_params()
    == return_model.get_params()
)


# Verify restored predictions across the ENTIRE selection period.
# No calibration or holdout observations are used.
with threadpool_limits(limits=TRAINING_SETTINGS["thread_limit"]):
    restored_probabilities = restored_bundle[
        "direction_model"
    ].predict_proba(X_selection)

    restored_return_predictions = restored_bundle[
        "return_model"
    ].predict(X_selection)

    restored_prior_probabilities = restored_bundle[
        "prior_classifier"
    ].predict_proba(X_selection)


np.testing.assert_allclose(
    restored_probabilities,
    selection_probabilities,
    rtol=1e-12,
    atol=1e-12,
)

np.testing.assert_allclose(
    restored_return_predictions,
    selection_return_predictions,
    rtol=1e-12,
    atol=1e-12,
)

np.testing.assert_allclose(
    restored_prior_probabilities,
    prior_probabilities,
    rtol=1e-12,
    atol=1e-12,
)

print(
    f"Model prediction round-trip: PASS "
    f"({len(X_selection):,} selection observations)"
)


def table_records(frame):
    # Convert pandas timestamps and numeric scalars into ordinary JSON.
    # Reports are human-readable summaries; model precision is retained
    # separately inside the binary model bundle.
    numeric_columns = frame.select_dtypes(include="number")

    assert np.isfinite(
        numeric_columns.to_numpy(dtype="float64")
    ).all(), "A report table contains non-finite numeric values."

    return json.loads(
        frame.to_json(
            orient="records",
            date_format="iso",
            date_unit="ns",
            double_precision=15,
        )
    )


baseline_report = {
    "experiment_id": BASELINE_EXPERIMENT_ID,
    "checkpoint_validation": "PASS",
    "research_status": BASELINE_RESEARCH_STATUS,
    "created_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "environment_versions": environment_versions,
    "source": {
        "dataset_relative_path": str(
            FEATURE_TARGET_INPUT_FILE.relative_to(PROJECT_ROOT)
        ),
        "dataset_sha256": MODEL_INPUT_SHA256,
        "dataset_build_status": dataset_manifest["status"],
        "feature_manifest_relative_path": str(
            FEATURE_TARGET_MANIFEST_FILE.relative_to(PROJECT_ROOT)
        ),
        "feature_manifest_fingerprint": feature_manifest_fingerprint,
    },
    "model_bundle": {
        "relative_path": str(
            BASELINE_MODEL_FILE.relative_to(PROJECT_ROOT)
        ),
        "sha256": file_sha256(TEMP_BASELINE_MODEL_FILE),
        "size_bytes": TEMP_BASELINE_MODEL_FILE.stat().st_size,
        "trusted_local_pickle_format": True,
    },
    "configuration": {
        "instrument": instrument_config,
        "features": feature_settings,
        "targets": target_settings,
        "splits": SPLIT_SETTINGS,
        "training": TRAINING_SETTINGS,
        "direction_model_parameters": direction_model.get_params(),
        "return_model_parameters": return_model.get_params(),
        "model_input_columns": MODEL_FEATURE_COLUMNS,
        "class_labels": [int(code) for code in CLASS_LABELS],
        "training_mean_return_bps": TRAINING_MEAN_RETURN_BPS,
    },
    "tables": {
        "split_summary": table_records(split_summary),
        "classification_metrics": table_records(classification_metrics),
        "regression_metrics": table_records(regression_metrics),
        "direction_class_report": table_records(direction_class_report),
        "selection_confusion_matrix": table_records(
            selection_confusion_matrix.reset_index().rename(
                columns={"index": "actual_direction"}
            )
        ),
        "training_selection_comparison": table_records(
            training_selection_comparison
        ),
        "probability_diagnostics": table_records(
            probability_diagnostics
        ),
        "return_ranking_diagnostics": table_records(
            return_ranking_diagnostics
        ),
    },
    "fit_timings_minutes": fit_timings,
    "raw_directional_probability_above_70_counts": (
        directional_probability_counts
    ),
    "validation": {
        "dataset_identity_verified_before_fitting": True,
        "chronological_splits_verified": True,
        "target_boundary_purging_verified": True,
        "model_input_whitelist_used": True,
        "internal_random_early_stopping_disabled": True,
        "restored_selection_predictions_match": True,
    },
    "limitations": [
        "This baseline has not demonstrated actionable directional edge.",
        "All 2023 argmax direction predictions were FLAT.",
        "Neither raw directional probability exceeded 70% in 2023.",
        "Return-estimation improvement over constant benchmarks was tiny.",
        "Return-bin averages are gross and based on overlapping observations.",
        "Probability calibration has not been fitted.",
        "Trading gates and executable P&L have not been evaluated.",
        "Calibration and holdout performance remain unevaluated.",
        "Original source coverage warnings remain.",
    ],
}


with TEMP_BASELINE_REPORT_FILE.open(
    "x", encoding="utf-8"
) as handle:
    json.dump(
        baseline_report,
        handle,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

with TEMP_BASELINE_REPORT_FILE.open(
    "r", encoding="utf-8"
) as handle:
    assert json.load(handle) == baseline_report


# Recheck before promoting the verified checkpoint files.
current_stat = FEATURE_TARGET_INPUT_FILE.stat()

assert (
    current_stat.st_size,
    current_stat.st_mtime_ns,
) == MODEL_INPUT_SNAPSHOT

assert not BASELINE_MODEL_FILE.exists()
assert not BASELINE_REPORT_FILE.exists()

TEMP_BASELINE_MODEL_FILE.rename(BASELINE_MODEL_FILE)
TEMP_BASELINE_REPORT_FILE.rename(BASELINE_REPORT_FILE)

assert BASELINE_MODEL_FILE.exists()
assert BASELINE_REPORT_FILE.exists()
assert not TEMP_BASELINE_MODEL_FILE.exists()
assert not TEMP_BASELINE_REPORT_FILE.exists()


print("Checkpoint report validation: PASS")
print("Baseline checkpoint: COMPLETE")
print("Research status:", BASELINE_RESEARCH_STATUS)
print("Model bundle:", BASELINE_MODEL_FILE)
print("Research report:", BASELINE_REPORT_FILE)
print("Calibration and holdout performance remain untouched.")


# Retain original models and diagnostic tables for comparison.
# Release only the duplicate restored models and predictions.
del restored_bundle, restored_probabilities
del restored_return_predictions, restored_prior_probabilities

Model prediction round-trip: PASS (329,184 selection observations)
Checkpoint report validation: PASS
Baseline checkpoint: COMPLETE
Research status: BASELINE_ONLY_NOT_TRADE_READY
Model bundle: C:\Users\Ijzd4\trend models\EURUSD 15\models\eurusd_baseline_h10_r5_v1.joblib
Research report: C:\Users\Ijzd4\trend models\EURUSD 15\reports\tables\eurusd_baseline_h10_r5_v1_report.json
Calibration and holdout performance remain untouched.


In [9]:
# Cell 6 — Test increased model capacity.
#
# Change exactly ONE model parameter:
# max_leaf_nodes: 15 -> 63.
#
# Keep training rows, features, targets, splits, learning rate,
# iteration count, regularization, and random seed unchanged.
#
# Create NEW model objects. Do not overwrite the saved baseline.
# Evaluate on 2023 only; calibration and holdout remain reserved.
#
# A matched training sample is used only for diagnostic scoring.
# Both candidate models are still fitted on ALL training rows.

CAPACITY_EXPERIMENT_ID = (
    f"{symbol_slug}_capacity63_h{PRIMARY_HORIZON_MINUTES}"
    f"_r{AUXILIARY_HORIZON_MINUTES}_v1"
)

CAPACITY_TREE_SETTINGS = TREE_SETTINGS.copy()
CAPACITY_TREE_SETTINGS["max_leaf_nodes"] = 63

assert TREE_SETTINGS["max_leaf_nodes"] == 15

changed_settings = {
    key
    for key in TREE_SETTINGS
    if TREE_SETTINGS[key] != CAPACITY_TREE_SETTINGS[key]
}

assert changed_settings == {"max_leaf_nodes"}


capacity_direction_model = HistGradientBoostingClassifier(
    loss="log_loss",
    class_weight=None,
    **CAPACITY_TREE_SETTINGS,
)

capacity_return_model = HistGradientBoostingRegressor(
    loss="squared_error",
    **CAPACITY_TREE_SETTINGS,
)


# Verify constructor parameters differ from the actual baseline
# only in the intended capacity setting.
for baseline_model, candidate_model in [
    (direction_model, capacity_direction_model),
    (return_model, capacity_return_model),
]:
    baseline_parameters = baseline_model.get_params()
    candidate_parameters = candidate_model.get_params()

    parameter_differences = {
        key
        for key in baseline_parameters
        if baseline_parameters[key] != candidate_parameters[key]
    }

    assert parameter_differences == {"max_leaf_nodes"}


current_stat = FEATURE_TARGET_INPUT_FILE.stat()

assert (
    current_stat.st_size,
    current_stat.st_mtime_ns,
) == MODEL_INPUT_SNAPSHOT

assert X_training.shape[1] == len(MODEL_FEATURE_COLUMNS)
assert X_selection.shape[1] == len(MODEL_FEATURE_COLUMNS)

print("Controlled-experiment preflight: PASS")
print("Changed parameter: max_leaf_nodes, 15 -> 63")
print(f"Training rows: {len(X_training):,}")
print(f"Model-selection rows: {len(X_selection):,}")


capacity_fit_timings = {}

with threadpool_limits(limits=TRAINING_SETTINGS["thread_limit"]):
    print("\nFitting capacity direction model...", flush=True)
    started = time.perf_counter()

    capacity_direction_model.fit(
        X_training, y_training_direction
    )

    capacity_fit_timings["direction_model_minutes"] = (
        time.perf_counter() - started
    ) / 60

    print("\nFitting capacity return model...", flush=True)
    started = time.perf_counter()

    capacity_return_model.fit(
        X_training, y_training_return
    )

    capacity_fit_timings["return_model_minutes"] = (
        time.perf_counter() - started
    ) / 60

    capacity_selection_probabilities = (
        capacity_direction_model.predict_proba(X_selection)
    )

    capacity_selection_return_predictions = (
        capacity_return_model.predict(X_selection)
    )


assert not capacity_direction_model.do_early_stopping_
assert not capacity_return_model.do_early_stopping_

assert (
    capacity_direction_model.n_iter_
    == CAPACITY_TREE_SETTINGS["max_iter"]
)
assert (
    capacity_return_model.n_iter_
    == CAPACITY_TREE_SETTINGS["max_iter"]
)

np.testing.assert_array_equal(
    capacity_direction_model.classes_, CLASS_LABELS
)


# Compare candidates with the unchanged benchmarks.
# Reuse the evaluation functions from Cell 3.
capacity_classification_metrics = pd.concat(
    [
        classification_metrics.assign(
            model=classification_metrics["model"].replace({
                "hist_gradient_boosting": "baseline_15_leaves"
            })
        ),
        pd.DataFrame([
            classification_metrics_row(
                "candidate_63_leaves",
                capacity_selection_probabilities,
            )
        ]),
    ],
    ignore_index=True,
)

capacity_regression_metrics = pd.concat(
    [
        regression_metrics.assign(
            model=regression_metrics["model"].replace({
                "hist_gradient_boosting": "baseline_15_leaves"
            })
        ),
        pd.DataFrame([
            regression_metrics_row(
                "candidate_63_leaves",
                capacity_selection_return_predictions,
            )
        ]),
    ],
    ignore_index=True,
)


# Matched training-sample diagnostics:
# score BOTH models on the same deterministic observations.
# These are in-sample diagnostics, not unseen performance.
training_sample_positions = np.linspace(
    0,
    len(X_training) - 1,
    num=min(100_000, len(X_training)),
    dtype=int,
)

sample_X = X_training[training_sample_positions]
sample_direction_y = y_training_direction[training_sample_positions]
sample_return_y = y_training_return[training_sample_positions]

training_sample_rows = []

with threadpool_limits(limits=TRAINING_SETTINGS["thread_limit"]):
    for name, classifier, regressor in [
        ("baseline_15_leaves", direction_model, return_model),
        (
            "candidate_63_leaves",
            capacity_direction_model,
            capacity_return_model,
        ),
    ]:
        sample_probabilities = classifier.predict_proba(sample_X)
        sample_return_predictions = regressor.predict(sample_X)

        sample_direction_predictions = CLASS_LABELS[
            np.argmax(sample_probabilities, axis=1)
        ]

        training_sample_rows.append({
            "model": name,
            "sample_rows": len(sample_X),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    sample_direction_y,
                    sample_direction_predictions,
                )
            ),
            "log_loss": float(
                log_loss(
                    sample_direction_y,
                    sample_probabilities,
                    labels=CLASS_LABELS,
                )
            ),
            "return_rmse_bps": float(
                np.sqrt(
                    mean_squared_error(
                        sample_return_y,
                        sample_return_predictions,
                    )
                )
            ),
            "return_r2": float(
                r2_score(
                    sample_return_y,
                    sample_return_predictions,
                )
            ),
        })

capacity_training_sample_metrics = pd.DataFrame(
    training_sample_rows
)


# Inspect candidate probability ranges and prediction frequencies.
capacity_direction_predictions = CLASS_LABELS[
    np.argmax(capacity_selection_probabilities, axis=1)
]

capacity_probability_rows = []

for column_index, code in enumerate(CLASS_LABELS):
    probabilities = capacity_selection_probabilities[:, column_index]

    capacity_probability_rows.append({
        "direction": CLASS_NAMES[int(code)],
        "mean_probability": float(probabilities.mean()),
        "p95": float(np.quantile(probabilities, 0.95)),
        "maximum": float(probabilities.max()),
        "argmax_prediction_count": int(
            np.sum(capacity_direction_predictions == code)
        ),
        "raw_probability_above_70_count": int(
            np.sum(probabilities > 0.70)
        ),
    })

capacity_probability_diagnostics = pd.DataFrame(
    capacity_probability_rows
)


# Inspect the candidate's gross-return ranking on 2023.
if np.unique(capacity_selection_return_predictions).size > 1:
    capacity_bins = pd.qcut(
        pd.Series(capacity_selection_return_predictions),
        q=10,
        labels=False,
        duplicates="drop",
    )
else:
    capacity_bins = pd.Series(
        np.zeros(len(X_selection), dtype="int8")
    )

capacity_return_frame = pd.DataFrame({
    "prediction_bin": capacity_bins,
    "predicted_return_bps": capacity_selection_return_predictions,
    "actual_return_bps": y_selection_return,
})

assert not capacity_return_frame.isna().any().any()

capacity_return_ranking = (
    capacity_return_frame
    .groupby("prediction_bin", sort=True, observed=True)
    .agg(
        observations=("actual_return_bps", "size"),
        mean_prediction_bps=("predicted_return_bps", "mean"),
        mean_actual_bps=("actual_return_bps", "mean"),
        actual_std_bps=("actual_return_bps", "std"),
    )
    .reset_index()
)

assert int(
    capacity_return_ranking["observations"].sum()
) == len(X_selection)


current_stat = FEATURE_TARGET_INPUT_FILE.stat()

assert (
    current_stat.st_size,
    current_stat.st_mtime_ns,
) == MODEL_INPUT_SNAPSHOT


print("\nCapacity-experiment pipeline checks: PASS")
print("Evaluation period: 2023 only")

print("\nClassification comparison:")
display(capacity_classification_metrics.round(6))

print("\nGross-return regression comparison:")
display(capacity_regression_metrics.round(6))

print("\nMatched training-sample comparison — IN SAMPLE:")
display(capacity_training_sample_metrics.round(6))

print("\nCandidate probability diagnostics:")
display(capacity_probability_diagnostics.round(6))

print("\nCandidate gross-return ranking:")
display(capacity_return_ranking.round(6))

print("\nCandidate fit timings:")
display(pd.DataFrame([capacity_fit_timings]).round(2))

print("Saved baseline remains unchanged.")
print("No candidate has been promoted or calibrated.")
print("Calibration and holdout performance remain unevaluated.")


# Retain candidate models and comparison tables.
# Release temporary diagnostic arrays only.
del sample_X, sample_direction_y, sample_return_y
del sample_probabilities, sample_return_predictions
del sample_direction_predictions, training_sample_positions
del capacity_bins, capacity_return_frame

Controlled-experiment preflight: PASS
Changed parameter: max_leaf_nodes, 15 -> 63
Training rows: 4,440,160
Model-selection rows: 329,184

Fitting capacity direction model...
Binning 0.533 GB of training data: 1.217 s
Fitting gradient boosted rounds:
[1/100] 3 trees, 189 leaves (63 on avg), max depth = 10, in 0.878s
[2/100] 3 trees, 189 leaves (63 on avg), max depth = 9, in 0.907s
[3/100] 3 trees, 189 leaves (63 on avg), max depth = 11, in 0.885s
[4/100] 3 trees, 189 leaves (63 on avg), max depth = 9, in 0.779s
[5/100] 3 trees, 189 leaves (63 on avg), max depth = 10, in 0.726s
[6/100] 3 trees, 189 leaves (63 on avg), max depth = 10, in 0.749s
[7/100] 3 trees, 189 leaves (63 on avg), max depth = 10, in 0.742s
[8/100] 3 trees, 189 leaves (63 on avg), max depth = 12, in 0.723s
[9/100] 3 trees, 189 leaves (63 on avg), max depth = 10, in 0.704s
[10/100] 3 trees, 189 leaves (63 on avg), max depth = 10, in 0.692s
[11/100] 3 trees, 189 leaves (63 on avg), max depth = 12, in 0.717s
[12/100] 3 tr

,model,accuracy,balanced_accuracy,macro_f1,log_loss,multiclass_brier_sum
0,training_class_prior,0.712018,0.333333,0.277263,0.800618,0.451964
1,baseline_15_leaves,0.712018,0.333333,0.277263,0.790625,0.446158
2,candidate_63_leaves,0.712018,0.333333,0.277263,0.790271,0.445967



Gross-return regression comparison:


,model,mae_bps,rmse_bps,r2,mean_prediction_bps,mean_error_bps
0,always_zero,1.915085,2.908988,-0.000000,0.000000,-0.000888
1,training_mean,1.915122,2.908990,-0.000001,-0.002037,-0.002925
2,baseline_15_leaves,1.914532,2.908647,0.000234,-0.001442,-0.002331
3,candidate_63_leaves,1.914721,2.908953,0.000024,0.000382,-0.000506



Matched training-sample comparison — IN SAMPLE:


...


Candidate gross-return ranking:


,prediction_bin,observations,mean_prediction_bps,mean_actual_bps,actual_std_bps
0,0,32919,-0.171121,-0.103407,4.018223
1,1,32926,-0.068381,-0.030307,2.968531
2,2,32910,-0.036823,-0.000335,2.635158
3,3,32921,-0.018864,-0.029341,2.371485
4,4,32916,-0.005049,0.033919,2.324217
5,5,32918,0.008622,0.002625,2.277021
6,6,32919,0.023473,0.014144,2.413514
7,7,32918,0.041588,0.021111,2.452044
8,8,32918,0.067266,0.025401,2.767723
9,9,32919,0.163115,0.075083,4.122182



Candidate fit timings:


,direction_model_minutes,return_model_minutes
0,1.04,0.22


Saved baseline remains unchanged.
No candidate has been promoted or calibrated.
Calibration and holdout performance remain unevaluated.
